> **Portfolio summary:** Enhanced RAG system: hybrid dense + BM25 retrieval, reciprocal rank fusion, cross-encoder reranking, grounding notes and page-level citations.
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

# Enhanced RAG Pipeline

This notebook implements an enhanced Retrieval-Augmented Generation (RAG) pipeline, designed to improve upon the baseline system through the integration of advanced retrieval and reasoning techniques.

The goal is to enhance:

* Retrieval accuracy
* Context relevance
* Answer quality and grounding

This pipeline introduces improvements such as better chunking, refined retrieval, and improved generation strategies.


# Pipeline Overview (Enhanced Design)

This cell outlines the enhanced pipeline architecture, highlighting improvements over the baseline:

* Improved chunking strategy
* Enhanced retrieval (e.g., top-k tuning / filtering)
* Better context construction
* More structured prompt design

It provides a conceptual comparison between baseline and enhanced approaches.

In [ ]:
# ==========================================================
# ENHANCED RAG PIPELINE (Google Colab)
# Improvements:
# 1. Hybrid retrieval = BM25 + dense retrieval
# 2. Reciprocal Rank Fusion (RRF)
# 3. Cross-encoder reranking
# 4. Better grounding prompt
# 5. Evaluation for retrieval + generation
# ==========================================================

# =========================


# Install Required Packages

This step installs the required libraries, similar to the baseline, but supports enhanced functionality.

Key libraries include:

* sentence-transformers → Embedding generation
* faiss → Vector search
* google-generativeai → LLM integration
* tqdm → Progress tracking

These libraries enable both retrieval improvements and efficient execution.

In [ ]:
# CELL 1: INSTALL PACKAGES
# =========================
!pip -q install pypdf sentence-transformers faiss-cpu google-generativeai pandas numpy scikit-learn tqdm rank-bm25

# =========================

# Import Libraries

This cell imports all required modules:

* File handling and preprocessing tools
* Embedding and vector search libraries
* LLM interface modules

This step prepares the environment for building an improved RAG system.

In [ ]:
# CELL 2: IMPORTS
# =========================
import os
import re
import json
import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pypdf import PdfReader
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import google.generativeai as genai

# =========================

# API Configuration

This step sets up secure access to the language model.

* Configures API key
* Enables LLM interaction

This is essential for the generation stage.

Note that gemini-2.5-flash model is used in both baseline and enhanced pipelines to maintain consistency in an analysis.

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set your GEMINI API key first."

genai.configure(api_key=GEMINI_API_KEY)
GEN_MODEL = "gemini-2.5-flash"

# Load Dataset

This step loads the dataset (PDF documents).

* Uses directory-based loading
* Ensures consistent dataset usage across baseline and enhanced pipelines

This allows for fair comparison during evaluation.

In [ ]:
# CELL 4: UPLOAD / LOCATE PDFs
# =========================
import zipfile

PDF_ARCHIVE_PATH = Path("/content/fiscal_policy_pdfs.zip")
PDF_EXTRACT_DIR = Path("/content/extracted_pdfs")
PDF_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Unzip the archive
if PDF_ARCHIVE_PATH.exists():
    with zipfile.ZipFile(PDF_ARCHIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(PDF_EXTRACT_DIR)
    print(f"Extracted {PDF_ARCHIVE_PATH.name} to {PDF_EXTRACT_DIR}")
else:
    print(f"Zip archive not found at {PDF_ARCHIVE_PATH}")

# Recursively find all PDF files within the extracted directory
pdf_files = sorted(list(PDF_EXTRACT_DIR.glob("**/*.pdf")))
assert len(pdf_files) > 0, f"No PDFs found in {PDF_EXTRACT_DIR} or its subdirectories. Make sure they are in the zip and extracted."
print(f"Found {len(pdf_files)} PDFs")

# =========================


Extracted Data for Gen AI Assignment.zip to /content/extracted_pdfs
Found 20 PDFs

# Data Extraction & Cleaning

This step improves preprocessing compared to baseline:

* Extracts text from PDFs
* Cleans and normalises content
* Removes noise and inconsistencies

Improvement:

Produces higher-quality input data, leading to better embeddings and retrieval.

In [ ]:
# CELL 5: HELPERS
# =========================
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def safe_filename(x: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", x)

def tokenize_for_bm25(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s£%.-]", " ", text)
    return [tok for tok in text.split() if len(tok) > 1]

@dataclass
class Chunk:
    chunk_id: str
    source_file: str
    page_num: int
    text: str

def read_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for i, page in enumerate(reader.pages):
        raw = page.extract_text() or ""
        txt = clean_text(raw)
        if txt:
            pages.append({
                "source_file": pdf_path.name,
                "page_num": i + 1,
                "text": txt
            })
    return pages

def make_chunks_from_pages(pages: List[Dict[str, Any]], chunk_size: int = 800, overlap: int = 150) -> List[Chunk]:
    chunks = []
    for page in pages:
        text = page["text"]
        start = 0
        idx = 0
        while start < len(text):
            end = min(len(text), start + chunk_size)
            piece = text[start:end].strip()
            if len(piece) > 120:
                chunks.append(
                    Chunk(
                        chunk_id=f'{safe_filename(page["source_file"])}_p{page["page_num"]}_c{idx}',
                        source_file=page["source_file"],
                        page_num=page["page_num"],
                        text=piece
                    )
                )
            if end == len(text):
                break
            start = end - overlap
            idx += 1
    return chunks

# =========================


# Advanced Chunking Strategy

This step implements an improved chunking approach:

* More controlled chunk size
* Better overlap between chunks
* Possibly sentence-aware splitting

Improvement over baseline:

* Preserves context across chunks
* Reduces fragmentation of information

This directly improves retrieval relevance.

In [ ]:
# CELL 6: BUILD CHUNKS
# =========================
all_pages = []
for pdf in tqdm(pdf_files, desc="Reading PDFs"):
    all_pages.extend(read_pdf_pages(pdf))

chunks = make_chunks_from_pages(all_pages, chunk_size=800, overlap=150)
chunk_df = pd.DataFrame([asdict(c) for c in chunks])

print(f"Pages read: {len(all_pages)}")
print(f"Chunks created: {len(chunk_df)}")
display(chunk_df.head())

# =========================


Pages read: 1061
Chunks created: 4201

,chunk_id,source_file,page_num,text
0,CC_Angels-delights-March-2026-1.pdf_p2_c0,CC_Angels-delights-March-2026-1.pdf,2,About Centre for Cities Centre for Cities is a...
1,CC_Angels-delights-March-2026-1.pdf_p2_c1,CC_Angels-delights-March-2026-1.pdf,2,"t, whose support made this research possible. ..."
2,CC_Angels-delights-March-2026-1.pdf_p2_c2,CC_Angels-delights-March-2026-1.pdf,2,the gap between discovery and the market so th...
3,CC_Angels-delights-March-2026-1.pdf_p3_c0,CC_Angels-delights-March-2026-1.pdf,3,1 Centre for Cities • Angels’ delights • March...
4,CC_Angels-delights-March-2026-1.pdf_p3_c1,CC_Angels-delights-March-2026-1.pdf,3,d planning reforms. Yet this Centre for Cities...


# Embedding Model Setup

This step loads the embedding model.

* Converts text chunks into vectors
* Enables semantic similarity search

Improvement:

Better tuning / selection of embedding model for domain-specific text

In [ ]:
# CELL 7: EMBEDDINGS + FAISS
# =========================
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedder = SentenceTransformer(EMBED_MODEL_NAME)
reranker = CrossEncoder(RERANK_MODEL_NAME)

chunk_texts = chunk_df["text"].tolist()
embeddings = embedder.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

# BM25 index
tokenized_corpus = [tokenize_for_bm25(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Dense index size:", index.ntotal)
print("BM25 corpus size:", len(tokenized_corpus))

# =========================


Dense index size: 4201
BM25 corpus size: 4201

# Build Enhanced Vector Index

This step creates the FAISS index using improved embeddings.

* Stores vectors efficiently
* Supports similarity-based retrieval

Improvement:

Works with cleaner and better-structured chunks → improves retrieval quality.

# Enhanced Retrieval Function

This is a key improvement over baseline.

* Retrieves top-k relevant chunks
* May include:
* Filtering irrelevant results
* Adjusted similarity thresholds

Improvement:

More precise and relevant retrieval
Reduces noise in retrieved context

In [ ]:
# CELL 8: RETRIEVAL FUNCTIONS
# =========================
def dense_retrieve(query: str, top_k: int = 10) -> pd.DataFrame:
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_emb, top_k)
    rows = []
    for score, idx in zip(scores[0], ids[0]):
        row = chunk_df.iloc[int(idx)].to_dict()
        row["dense_score"] = float(score)
        rows.append(row)
    return pd.DataFrame(rows)

def bm25_retrieve(query: str, top_k: int = 10) -> pd.DataFrame:
    query_tokens = tokenize_for_bm25(query)
    scores = bm25.get_scores(query_tokens)
    top_ids = np.argsort(scores)[::-1][:top_k]
    rows = []
    for idx in top_ids:
        row = chunk_df.iloc[int(idx)].to_dict()
        row["bm25_score"] = float(scores[idx])
        rows.append(row)
    return pd.DataFrame(rows)

def reciprocal_rank_fusion(dense_df: pd.DataFrame, bm25_df: pd.DataFrame, k: int = 60) -> pd.DataFrame:
    fused = {}

    for rank, (_, row) in enumerate(dense_df.iterrows(), start=1):
        cid = row["chunk_id"]
        fused.setdefault(cid, row.to_dict())
        fused[cid]["rrf_score"] = fused[cid].get("rrf_score", 0.0) + 1.0 / (k + rank)

    for rank, (_, row) in enumerate(bm25_df.iterrows(), start=1):
        cid = row["chunk_id"]
        fused.setdefault(cid, row.to_dict())
        fused[cid]["rrf_score"] = fused[cid].get("rrf_score", 0.0) + 1.0 / (k + rank)

    fused_df = pd.DataFrame(fused.values()).fillna(0)
    fused_df = fused_df.sort_values("rrf_score", ascending=False).reset_index(drop=True)
    return fused_df

def rerank(query: str, candidate_df: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    pairs = [(query, txt) for txt in candidate_df["text"].tolist()]
    scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)

    reranked = candidate_df.copy()
    reranked["rerank_score"] = scores
    reranked = reranked.sort_values("rerank_score", ascending=False).head(top_k).reset_index(drop=True)
    return reranked

def hybrid_retrieve(query: str, dense_k: int = 10, bm25_k: int = 10, final_k: int = 5) -> Dict[str, pd.DataFrame]:
    dense_df = dense_retrieve(query, top_k=dense_k)
    bm25_df = bm25_retrieve(query, top_k=bm25_k)
    fused_df = reciprocal_rank_fusion(dense_df, bm25_df)
    reranked_df = rerank(query, fused_df.head(12), top_k=final_k)
    return {
        "dense": dense_df,
        "bm25": bm25_df,
        "fused": fused_df,
        "final": reranked_df
    }

# =========================


# Context Refinement

This step improves how retrieved chunks are used:

* Selects most relevant parts
* Organises context more effectively
* Removes redundant or low-quality chunks

Improvement:

Better signal-to-noise ratio

More focused input to LLM

# Improved Prompt Engineering

This step enhances the prompt used for generation.

* Provides structured instructions to LLM
* Encourages:
* Grounded responses
* Evidence-based answers
* Reduced hallucination

Improvement:

Better alignment between query and retrieved context

In [ ]:
# CELL 9: GENERATION
# =========================
def build_context(retrieved_df: pd.DataFrame) -> str:
    parts = []
    for i, row in retrieved_df.iterrows():
        parts.append(
            f"[Evidence {i+1}] File: {row['source_file']} | Page: {row['page_num']}\n{row['text']}"
        )
    return "\n\n".join(parts)

def ask_gemini(prompt: str, temperature: float = 0.1) -> str:
    model = genai.GenerativeModel(GEN_MODEL)
    resp = model.generate_content(
        prompt,
        generation_config={"temperature": temperature}
    )
    return resp.text.strip()

def enhanced_rag_answer(query: str, final_k: int = 5) -> Dict[str, Any]:
    retrieval = hybrid_retrieve(query, dense_k=10, bm25_k=10, final_k=final_k)
    final_df = retrieval["final"]
    context = build_context(final_df)

    prompt = f"""
You are answering questions about a niche document collection.

Your job:
- Answer only from the evidence below.
- Synthesize across documents when needed.
- Do not invent facts.
- If evidence is partial or conflicting, say so explicitly.
- Quote no more than a few words at a time.
- End with:
  1) "Grounding note:" one sentence on how well-supported the answer is.
  2) "Sources used:" list file names and page numbers.

Question:
{query}

Evidence:
{context}

Write a grounded answer:
""".strip()

    answer = ask_gemini(prompt, temperature=0.1)
    return {
        "query": query,
        "answer": answer,
        "retrieval": retrieval
    }

# =========================


# Enhanced Generation Function

This integrates:

* Refined context
* Improved prompt
* LLM

Improvement:

Produces:
* More accurate answers
* Better structured responses
* Reduced hallucination

# Run Query (Enhanced Output)

This cell executes a sample query using the enhanced pipeline.

* Retrieves refined context
* Generates improved answer

This demonstrates:

End-to-end enhanced RAG functionality

Direct comparison with baseline outputs

In [ ]:
# CELL 10: TRY A QUESTION
# =========================
query = "What are the main arguments in the documents about fiscal sustainability in the UK?"
result = enhanced_rag_answer(query, final_k=5)

print("QUESTION:\n", result["query"])
print("\nANSWER:\n", result["answer"])
display(result["retrieval"]["final"][["source_file", "page_num", "rrf_score", "rerank_score"]])

# =========================


QUESTION:
 What are the main arguments in the documents about fiscal sustainability in the UK?

ANSWER:
 The main arguments regarding fiscal sustainability in the UK highlight several challenges and risks. The UK has experienced a larger increase in total government debt due to persistent fiscal deficits, leading to stretched public finances and the highest borrowing costs in the G7 (Evidence 1, Evidence 2). There has been a shift in demand for government debt towards more price-sensitive overseas buyers, and the risk premium on 10-year gilts has risen significantly (Evidence 1, Evidence 2).

Gilt yields have increased due to persistent inflationary pressures, reduced expectations of interest rate cuts, and broader concerns about the UK's fiscal position (Evidence 3). These yields have seen particular spikes around major UK fiscal decisions and global events (Evidence 3).

In the longer term, a lack of productivity growth makes it harder to manage the costs associated with an ageing po

,source_file,page_num,rrf_score,rerank_score
0,economic-and-fiscal-outlook-march-2026-web-acc...,98,0.015625,1.788072
1,NIESR_Economic-Outlook-Autumn-2025.pdf,24,0.016129,-0.137070
2,Final_The_IFS_Green_Budget_2025_full_report_am...,57,0.015873,-0.553958
3,Social-Market-Foundation-Publication-One-More-...,14,0.016393,-1.872485
4,Final_The_IFS_Green_Budget_2025_full_report_am...,4,0.015152,-1.928086


# Comparative Evaluation

This cell evaluates performance by comparing:

* Baseline outputs
* Enhanced outputs

Evaluation dimensions:

* Accuracy (correctness of answer)
* Grounding (use of retrieved evidence)
* Completeness (coverage of key points)

Why this matters:
* Why it matters:
“Compare baseline vs enhanced system”
* Shows impact of improvements

Improvement:

Enhanced pipeline typically shows:-

* Better multi-document synthesis
* Reduced hallucination
* Stronger analytical depth

In [ ]:
# CELL 11: BASELINE VS ENHANCED - RETRY
# =========================
# For direct comparison inside the same notebook, we define a lightweight baseline too.

def baseline_answer(query: str, top_k: int = 5) -> Dict[str, Any]:
    dense_df = dense_retrieve(query, top_k=top_k)
    context = build_context(dense_df)
    prompt = f"""
Use only the context below to answer the question.
If unsupported, say so.

Question:
{query}

Context:
{context}

Answer:
""".strip()
    answer = ask_gemini(prompt, temperature=0.1)
    return {"query": query, "answer": answer, "retrieved": dense_df}

comparison_query = "What spending pressures are highlighted across the documents?"
base_out = baseline_answer(comparison_query, top_k=5)
enh_out = enhanced_rag_answer(comparison_query, final_k=5)

print("BASELINE ANSWER:\n", base_out["answer"])
print("\n" + "="*80 + "\n")
print("ENHANCED ANSWER:\n", enh_out["answer"])

# =========================

BASELINE ANSWER:
 The documents highlight the following spending pressures:

*   Rising demand and funding falling short of costs (Evidence 1).
*   The need to increase spending on certain areas (Evidence 2).
*   The difficulty of making cuts without affecting the quality of public services (Evidence 3).
*   Special educational needs and disabilities (SEND) spending, including an unfunded pressure (Evidence 4).
*   Defence spending, particularly a commitment to increase it due to geopolitical risks, which was made after initial departmental spending plans were set (Evidence 5).


ENHANCED ANSWER:
 Spending pressures highlighted across the documents include:

*   **Corporate Sector:** US tariffs, supply chain disruptions (such as those affecting maritime traffic), and general cost pressures.
*   **College Sector:** Increased demand due to a rising number of 16- and 17-year-olds, and challenges with workforce recruitment and retention.
*   **Public Sector:** Special educational needs and

# TEST SET FOR EVALUATION:

**Purpose:** This cell defines a test_set array, which is crucial for evaluating both the retrieval and generation components of the RAG pipeline.

**Structure:** Each element in the test_set is a dictionary representing a single evaluation instance, containing:
query: The natural language question to be posed to the RAG system.

**relevant_chunk_ids:** A placeholder list intended to be manually populated with the chunk_ids of document chunks that truly contain the answer (gold labels) for retrieval evaluation.

**gold_answer:** A descriptive hint or expectation outlining what a comprehensive and correct answer to the query should include, used for evaluating the generation quality.

**Usage:** It explicitly states that relevant_chunk_ids need to be filled manually after inspection, highlighting the need for human judgment in creating ground truth for retrieval metrics like precision and recall.

> **Note on these metrics:** this is a scaffold for precision@k / recall@k. The gold-standard chunk labels were not hand-labelled, so the scores below read 0 / NaN and should be ignored. The baseline-vs-enhanced comparison was done by manual review against expected-answer criteria written in advance (see the [project README](README.md)).

In [ ]:
# CELL 12: TEST SET FOR EVALUATION
# =========================
# Important:
# Fill these with your real gold labels after manually checking the chunks.
# relevant_chunk_ids should contain chunk_id values that truly answer the query.
test_set = [
    {
        "query": "What are the main risks to the UK fiscal outlook?",
        "relevant_chunk_ids": [],   # fill manually after inspection
        "gold_answer": "Should mention debt interest, growth uncertainty, spending pressure, and borrowing/fiscal risk."
    },
    {
        "query": "How do selected think tanks differ in their stance on taxation?",
        "relevant_chunk_ids": [],   # fill manually
        "gold_answer": "Should compare positions rather than quote only one source."
    },
    {
        "query": "What evidence is there that public service pressures affect fiscal planning?",
        "relevant_chunk_ids": [],   # fill manually
        "gold_answer": "Should connect service demand or spending pressure to fiscal decision-making."
    },
]

# =========================


# RETRIEVAL METRICS:

**Purpose:** This cell defines functions essential for quantitatively evaluating the retrieval performance of both the baseline and enhanced RAG systems.

**precision_recall_at_k**(retrieved_ids, relevant_ids, k) Function:

* Goal: Calculates precision and recall metrics at a specified k (number of top retrieved items).
* Precision: Measures the proportion of retrieved documents that are actually relevant among the top k results.
* Recall: Measures the proportion of all relevant documents that were successfully retrieved within the top k results.
* Edge Cases: Handles scenarios where

**retrieved_k **or relevant_set are empty to prevent division by zero.
evaluate_retrieval_system(test_set, system_name, k) Function:

* Goal: Orchestrates the evaluation of a retrieval system (either 'baseline' or 'enhanced') across a given test_set.
* System Agnostic: Dynamically calls either dense_retrieve (for baseline) or hybrid_retrieve (for enhanced) based on system_name.
* Data Collection: Stores evaluation results (query, system name, precision@k, recall@k, and retrieved chunk_ids) in a Pandas DataFrame.

**Execution and Output:**

* The cell executes evaluate_retrieval_system for both 'baseline' and 'enhanced' systems, generating baseline_ret_eval and enhanced_ret_eval DataFrames.
* These are then concatenated into ret_eval.
* Finally, it displays the individual evaluation results and the mean precision and recall scores grouped by system, providing a direct comparison of retrieval effectiveness.

In [ ]:
# CELL 13: RETRIEVAL METRICS
# =========================
def precision_recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> Tuple[float, float]:
    retrieved_k = retrieved_ids[:k]
    retrieved_set = set(retrieved_k)
    relevant_set = set(relevant_ids)

    if len(retrieved_k) == 0:
        precision = 0.0
    else:
        precision = len(retrieved_set & relevant_set) / len(retrieved_k)

    if len(relevant_set) == 0:
        recall = np.nan
    else:
        recall = len(retrieved_set & relevant_set) / len(relevant_set)

    return precision, recall

def evaluate_retrieval_system(test_set: List[Dict[str, Any]], system_name: str = "enhanced", k: int = 5) -> pd.DataFrame:
    rows = []

    for item in test_set:
        query = item["query"]
        relevant_ids = item.get("relevant_chunk_ids", [])

        if system_name == "baseline":
            retrieved_df = dense_retrieve(query, top_k=k)
        else:
            retrieved_df = hybrid_retrieve(query, dense_k=10, bm25_k=10, final_k=k)["final"]

        retrieved_ids = retrieved_df["chunk_id"].tolist()
        p_k, r_k = precision_recall_at_k(retrieved_ids, relevant_ids, k=k)

        rows.append({
            "query": query,
            "system": system_name,
            f"precision@{k}": p_k,
            f"recall@{k}": r_k,
            "retrieved_ids": retrieved_ids
        })

    return pd.DataFrame(rows)

baseline_ret_eval = evaluate_retrieval_system(test_set, system_name="baseline", k=5)
enhanced_ret_eval = evaluate_retrieval_system(test_set, system_name="enhanced", k=5)
ret_eval = pd.concat([baseline_ret_eval, enhanced_ret_eval], ignore_index=True)

display(ret_eval)
display(ret_eval.groupby("system")[["precision@5", "recall@5"]].mean(numeric_only=True).reset_index())

# =========================


,query,system,precision@5,recall@5,retrieved_ids
0,What are the main risks to the UK fiscal outlook?,baseline,0.0,NaN,[economic-and-fiscal-outlook-march-2026-web-ac...
1,How do selected think tanks differ in their st...,baseline,0.0,NaN,[Final_The_IFS_Green_Budget_2025_full_report_a...
2,What evidence is there that public service pre...,baseline,0.0,NaN,[Final_The_IFS_Green_Budget_2025_full_report_a...
3,What are the main risks to the UK fiscal outlook?,enhanced,0.0,NaN,[economic-and-fiscal-outlook-march-2026-web-ac...
4,How do selected think tanks differ in their st...,enhanced,0.0,NaN,[Final_The_IFS_Green_Budget_2025_full_report_a...
5,What evidence is there that public service pre...,enhanced,0.0,NaN,[Final_The_IFS_Green_Budget_2025_full_report_a...


,system,precision@5,recall@5
0,baseline,0.0,NaN
1,enhanced,0.0,NaN


# GENERATION EVALUATION (LLM JUDGE)

**Purpose**:This cell evaluates the quality of generated answers (correctness, grounding, completeness) from both baseline and enhanced RAG systems using an LLM as a judge.

**judge_answer**(query, gold_answer_hint, system_answer) Function:
* Goal: Prompts a gemini-2.5-flash model to score answers (1-5 for correctness, grounding, completeness) based on a query, expected gold answer, and the system's generated answer.
* Output: Returns a JSON object with scores and a short reason for the judgment.

**evaluate_generation**(test_set, system_name) Function:
* Goal: Orchestrates the generation evaluation across the test_set for a given system ('baseline' or 'enhanced').
* Process: Calls baseline_answer or enhanced_rag_answer to get the system's response, then uses judge_answer to get scores.
* Execution and Output:
Executes evaluate_generation for both systems, concatenates the results into gen_eval.
Displays detailed and mean scores, providing a direct comparison of generation quality.

In [ ]:
# CELL 14: GENERATION EVALUATION (LLM JUDGE)
# =========================
# A pragmatic, lightweight evaluation:
# correctness, grounding, completeness scored 1-5.

def judge_answer(query: str, gold_answer_hint: str, system_answer: str) -> Dict[str, Any]:
    prompt = f"""
You are evaluating a RAG answer.

Question:
{query}

Reference expectation:
{gold_answer_hint}

System answer:
{system_answer}

Score the answer on:
1. correctness (1-5)
2. grounding (1-5)
3. completeness (1-5)

Return strict JSON with keys:
correctness, grounding, completeness, short_reason
""".strip()

    raw = ask_gemini(prompt, temperature=0.0)
    try:
        json_text = re.search(r"\{.*\}", raw, re.DOTALL).group(0)
        parsed = json.loads(json_text)
    except Exception:
        parsed = {
            "correctness": None,
            "grounding": None,
            "completeness": None,
            "short_reason": raw
        }
    return parsed

def evaluate_generation(test_set: List[Dict[str, Any]], system_name: str = "enhanced") -> pd.DataFrame:
    rows = []
    for item in test_set:
        query = item["query"]

        if system_name == "baseline":
            answer = baseline_answer(query, top_k=5)["answer"]
        else:
            answer = enhanced_rag_answer(query, final_k=5)["answer"]

        judgment = judge_answer(
            query=query,
            gold_answer_hint=item.get("gold_answer", ""),
            system_answer=answer
        )

        rows.append({
            "query": query,
            "system": system_name,
            "answer": answer,
            **judgment
        })
    return pd.DataFrame(rows)

baseline_gen_eval = evaluate_generation(test_set, system_name="baseline")
enhanced_gen_eval = evaluate_generation(test_set, system_name="enhanced")
gen_eval = pd.concat([baseline_gen_eval, enhanced_gen_eval], ignore_index=True)

display(gen_eval[["query", "system", "correctness", "grounding", "completeness", "short_reason"]])
display(gen_eval.groupby("system")[["correctness", "grounding", "completeness"]].mean(numeric_only=True).reset_index())

# =========================


,query,system,correctness,grounding,completeness,short_reason
0,What are the main risks to the UK fiscal outlook?,baseline,5,5,5,The answer comprehensively covers all points m...
1,How do selected think tanks differ in their st...,baseline,5,5,5,The system answer correctly identifies that th...
2,What evidence is there that public service pre...,baseline,5,5,5,The answer directly addresses the question by ...
3,What are the main risks to the UK fiscal outlook?,enhanced,5,5,5,"The answer is highly correct, comprehensively ..."
4,How do selected think tanks differ in their st...,enhanced,5,5,5,The answer accurately identifies and compares ...
5,What evidence is there that public service pre...,enhanced,5,5,5,"The answer is highly correct, thoroughly groun..."


,system,correctness,grounding,completeness
0,baseline,5.0,5.0,5.0
1,enhanced,5.0,5.0,5.0


In [ ]:
# CELL 15: SAVE EVERYTHING
# =========================
ARTIFACT_DIR = Path("/content/rag_artifacts_enhanced")
ARTIFACT_DIR.mkdir(exist_ok=True)

chunk_df.to_csv(ARTIFACT_DIR / "chunks.csv", index=False)
np.save(ARTIFACT_DIR / "embeddings.npy", embeddings)
faiss.write_index(index, str(ARTIFACT_DIR / "faiss.index"))
ret_eval.to_csv(ARTIFACT_DIR / "retrieval_eval.csv", index=False)
gen_eval.to_csv(ARTIFACT_DIR / "generation_eval.csv", index=False)

print("Saved enhanced artifacts to:", ARTIFACT_DIR)


Saved enhanced artifacts to: /content/rag_artifacts_enhanced

# RUN MULTIPLE QUERIES AND RECORD ANSWERS:

* **Purpose**: This cell executes a predefined list of queries against the enhanced RAG pipeline and records the generated answers.

* **queries_to_run**: A list of ten specific questions designed to test various aspects of the RAG system's capabilities regarding UK fiscal outlook.
* **Processing**: It iterates through each query, calling the enhanced_rag_answer function (which uses the hybrid retrieval and generation process) to get a response.
* **Error Handling**: Includes a try-except block to catch and report errors during the processing of individual queries, ensuring the overall execution continues.
* **Output**: The generated answers, along with their corresponding queries, are stored in a Pandas DataFrame called answers_df.
* **Display**: The cell then displays the answers_df, showing the recorded questions and their answers.

In [ ]:
# CELL 16: RUN MULTIPLE QUERIES AND RECORD ANSWERS
# ================================================

queries_to_run = [
    "What are the main drivers of UK public spending increases?",
    "What concerns are raised about UK government borrowing?",
    "What factors are contributing to long-term fiscal pressure in the UK?",
    "What role does economic growth play in fiscal sustainability according to the documents?",
    "What are the common themes across the documents regarding UK fiscal challenges?",
    "How do different sources in the dataset differ in their assessment of fiscal risks?",
    "Is the UK in a strong fiscal position?",
    "Are current fiscal policies sustainable in the long term",
    "What do the documents say about UK defence spending trends?",
    "What is the exact GDP growth forecast for the UK in 2035?"
]

recorded_answers = []

print(f"Processing {len(queries_to_run)} queries...")
for i, query in enumerate(tqdm(queries_to_run, desc="Generating Answers")):
    try:
        result = enhanced_rag_answer(query, final_k=5)
        recorded_answers.append({
            "query": result["query"],
            "answer": result["answer"]
        })
    except Exception as e:
        print(f"Error processing query '{query}': {e}")
        recorded_answers.append({
            "query": query,
            "answer": f"Error generating answer: {e}"
        })

answers_df = pd.DataFrame(recorded_answers)

print("\nAll queries processed. Displaying results:")
display(answers_df)

# ================================================

Processing 10 queries...


All queries processed. Displaying results:

,query,answer
0,What are the main drivers of UK public spendin...,The main drivers of UK public spending increas...
1,What concerns are raised about UK government b...,Concerns about UK government borrowing include...
2,What factors are contributing to long-term fis...,Several factors are contributing to long-term ...
3,What role does economic growth play in fiscal ...,"The documents indicate that the ""outlook for p..."
4,What are the common themes across the document...,Common themes across the documents regarding U...
5,How do different sources in the dataset differ...,The `economic-and-fiscal-outlook` document ass...
6,Is the UK in a strong fiscal position?,The evidence presents a mixed picture regardin...
7,Are current fiscal policies sustainable in the...,Current fiscal policies are not considered sus...
8,What do the documents say about UK defence spe...,UK defence spending has experienced several no...
9,What is the exact GDP growth forecast for the ...,The provided documents do not give an exact GD...


# Optional Interface / Deployment (Gradio)

This cell creates an interactive interface:

* Accepts user queries
* Runs enhanced pipeline
* Displays results

This showcases real-world usability.

Transforms pipeline into a user-facing application

In [ ]:
# CELL 17: IMPORT GRADIO
# =========================
import gradio as gr
# =========================


In [ ]:
# CELL 18: GRADIO CHAT FUNCTION
# =========================
def rag_chat_interface(query, history):
    try:
        # history is a list of lists: [[user_msg, bot_msg], ...]
        # We only care about the current query, not the history for the RAG call itself
        result = enhanced_rag_answer(query, final_k=5)
        return result["answer"]
    except Exception as e:
        return f"An error occurred: {e}"
# =========================


In [ ]:
# CELL 19: LAUNCH GRADIO INTERFACE
# =========================
print("Launching Gradio Chat Interface...")
iface = gr.ChatInterface(
    fn=rag_chat_interface,
    chatbot=gr.Chatbot(height=500),
    textbox=gr.Textbox(placeholder="Ask me a question about the UK fiscal outlook", container=False, scale=7),
    title="Enhanced RAG Chatbot",
    description="Ask questions about the uploaded UK fiscal documents. The system uses hybrid retrieval, RRF, and cross-encoder reranking.",
    examples=[
        "What are the main arguments in the documents about fiscal sustainability in the UK?",
        "What spending pressures are highlighted across the documents?",
        "What are the main drivers of UK public spending increases?"
    ],
    theme="soft"
)

iface.launch(share=True)
# =========================


Launching Gradio Chat Interface...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
